## Introduction

This part trains the general-domain baseline that the rest of the project is compared against. We fine-tune `bert-base-cased` for disease named entity recognition on the NCBI Disease Corpus and report strict entity-level precision, recall and F1 on the test split. BioBERT (Part 4) is trained with the same code and the same hyperparameters, so any difference between the two comes from the pre-trained weights and not from the training setup. The DAPT experiment in Part 6 also starts from this model.

The notebook has five sections: setup, data inspection, tokenisation and label alignment, model and training, and results with a short discussion.

## Setup

We install the libraries used by every part of the project and load the NCBI Disease Corpus from the Hugging Face Hub. The corpus comes with fixed train, validation and test splits (5,433, 924 and 941 sentences), and we use them as they are so that all models are evaluated on the same data. The notebook was run with `datasets` 3.6.0.

In [3]:
# Install PyTorch, Hugging Face Datasets/Transformers and the seqeval metric (already present in the local environment).
!pip install torch datasets transformers evaluate seqeval

In [4]:
# Load the NCBI Disease Corpus with its official train/validation/test splits.
from datasets import load_dataset

dataset = load_dataset("ncbi/ncbi_disease")

# Show the size of each split
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 5433
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 924
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 941
    })
})


## Data inspection

Before training we look at one example to check the data format. Each sentence is already split into words, and each word carries one of three BIO tags: `O` for words outside a disease mention, `B-Disease` for the first word of a mention and `I-Disease` for the following words of the same mention. The first training sentence contains the four-word mention "adenomatous polyposis coli tumour".

In [6]:
# Print the id, words and integer tags of the first training sentence.
example = dataset["train"][0]

print("ID:")
print(example["id"])

print("\nTokens:")
print(example["tokens"])

print("\nNER tags:")
print(example["ner_tags"])

ID:
0

Tokens:
['Identification', 'of', 'APC2', ',', 'a', 'homologue', 'of', 'the', 'adenomatous', 'polyposis', 'coli', 'tumour', 'suppressor', '.']

NER tags:
[0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 0, 0]


In [7]:
# Map the integer tags to their names: 0 = O, 1 = B-Disease, 2 = I-Disease.
label_names = dataset["train"].features["ner_tags"].feature.names

print(label_names)

['O', 'B-Disease', 'I-Disease']


In [8]:
# Show each word next to its tag name to see where the disease mention starts and ends.
for token, tag in zip(example["tokens"], example["ner_tags"]):
    print(f"{token:20} {label_names[tag]}")

Identification       O
of                   O
APC2                 O
,                    O
a                    O
homologue            O
of                   O
the                  O
adenomatous          B-Disease
polyposis            I-Disease
coli                 I-Disease
tumour               I-Disease
suppressor           O
.                    O


## Tokenisation and label alignment

BERT splits words into WordPiece subwords, so the word-level tags no longer line up with the model inputs. We first show how `bert-base-cased` splits the example sentence, and then align the labels: the first subword of each word keeps the word's tag, while later subwords and the special tokens `[CLS]` and `[SEP]` get the label −100, which the loss function ignores. This way every word is predicted and scored exactly once.

In [9]:
# Load the cased WordPiece tokenizer that matches the bert-base-cased checkpoint.
from transformers import AutoTokenizer

model_name = "bert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print(tokenizer)

BertTokenizer(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


In [10]:
# Tokenise the example sentence; is_split_into_words=True tells the tokenizer the input is already split into words.
tokens = example["tokens"]

tokenized = tokenizer(
    tokens,
    is_split_into_words=True
)

# Convert the ids back to subword strings to see how each word was split
bert_tokens = tokenizer.convert_ids_to_tokens(tokenized["input_ids"])

print(bert_tokens)

['[CLS]', 'I', '##dent', '##ification', 'of', 'AP', '##C', '##2', ',', 'a', 'ho', '##mo', '##logue', 'of', 'the', 'ad', '##eno', '##mat', '##ous', 'p', '##oly', '##po', '##sis', 'co', '##li', 't', '##umour', 'suppress', '##or', '.', '[SEP]']


In [11]:
# word_ids() gives, for each subword, the index of the word it came from (None for [CLS] and [SEP]).
word_ids = tokenized.word_ids()

for bert_token, word_id in zip(bert_tokens, word_ids):
    print(f"{bert_token:15} {word_id}")

[CLS]           None
I               0
##dent          0
##ification     0
of              1
AP              2
##C             2
##2             2
,               3
a               4
ho              5
##mo            5
##logue         5
of              6
the             7
ad              8
##eno           8
##mat           8
##ous           8
p               9
##oly           9
##po            9
##sis           9
co              10
##li            10
t               11
##umour         11
suppress        12
##or            12
.               13
[SEP]           None


In [12]:
# Tokenise a batch of sentences and give each subword a training label.
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        label_ids = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None:
                # Special token ([CLS], [SEP]): ignored by the loss
                label_ids.append(-100)
            elif word_id != previous_word_id:
                # First subword of a word: keep the word's BIO tag
                label_ids.append(labels[word_id])
            else:
                # Later subword of the same word: ignored by the loss
                label_ids.append(-100)

            previous_word_id = word_id

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels

    return tokenized_inputs

In [13]:
# Check the alignment on the first sentence: IGNORE marks positions labelled -100.
test_data = dataset["train"].select([0])

test_result = tokenize_and_align_labels(test_data)

test_tokens = tokenizer.convert_ids_to_tokens(
    test_result["input_ids"][0]
)

test_labels = test_result["labels"][0]

for token, label in zip(test_tokens, test_labels):
    if label == -100:
        label_name = "IGNORE"
    else:
        label_name = label_names[label]

    print(f"{token:15} {label_name}")

[CLS]           IGNORE
I               O
##dent          IGNORE
##ification     IGNORE
of              O
AP              O
##C             IGNORE
##2             IGNORE
,               O
a               O
ho              O
##mo            IGNORE
##logue         IGNORE
of              O
the             O
ad              B-Disease
##eno           IGNORE
##mat           IGNORE
##ous           IGNORE
p               I-Disease
##oly           IGNORE
##po            IGNORE
##sis           IGNORE
co              I-Disease
##li            IGNORE
t               I-Disease
##umour         IGNORE
suppress        O
##or            IGNORE
.               O
[SEP]           IGNORE


In [14]:
# Apply the alignment to all three splits; this adds input_ids, attention_mask and labels.
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5433
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 924
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 941
    })
})


## Model and training

We put a linear token-classification head with three outputs on top of the pre-trained encoder and fine-tune all weights. The training settings are shared with BioBERT in Part 4: learning rate 2e-5, batch size 8, weight decay 0.01 and 3 epochs, with the validation set scored after every epoch. The data collator pads each batch to its longest sentence. The metric is seqeval in its default mode, which counts a predicted entity as correct only when its span matches the gold span exactly, and word-level accuracy is reported alongside it.

In [15]:
# Pre-trained bert-base-cased encoder plus a new 3-way classification head (its weights start at random).
from transformers import AutoModelForTokenClassification

# Label maps stored in the model config so predictions come back as tag names
id2label = {
    0: "O",
    1: "B-Disease",
    2: "I-Disease"
}

label2id = {
    "O": 0,
    "B-Disease": 1,
    "I-Disease": 2
}

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print(model.__class__.__name__)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 11293.97it/s]
[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not

BertForTokenClassification


In [16]:
# Pad input_ids and labels to the longest sentence in each batch (labels are padded with -100).
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [17]:
# Entity-level metrics with seqeval.
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p

    # Take the most likely tag for every subword
    predictions = np.argmax(predictions, axis=2)

    # Keep only positions with a real label (first subword of each word) and turn ids into tag names
    true_predictions = [
        [label_names[p] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_names[l] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

In [18]:
# Fine-tuning hyperparameters, identical to the BioBERT run in Part 4.
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert-ner-baseline",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",          # score the validation set after every epoch
    save_strategy="epoch",
    logging_steps=50,
    report_to="none"                # no external experiment tracker
)

In [19]:
# Hugging Face Trainer: trains on the train split and evaluates on the validation split.
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [21]:
# Fine-tune for 3 epochs (about 8 minutes on an Apple-silicon GPU); the table shows validation scores per epoch.
train_result = trainer.train()

/opt/anaconda3/envs/neuralnet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.041484,0.051802,0.777647,0.839898,0.807575,0.983896
2,0.025029,0.051599,0.770455,0.861499,0.813437,0.984522
3,0.023125,0.060924,0.798824,0.862770,0.829566,0.985523


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.83it/s]
/opt/anaconda3/envs/neuralnet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.12it/s]
/opt/anaconda3/envs/neuralnet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.79it/s]


## Results

During training the validation F1 rose from 80.76% after the first epoch to 81.34% and then 82.96% after the third, while the validation loss was lowest after the second epoch. We keep the model from the end of the third epoch and score it once on the test split below.

In [24]:
# Score the final model on the held-out test split (strict entity-level P/R/F1 and word-level accuracy).
test_results = trainer.evaluate(tokenized_dataset["test"])

print("Precision:", test_results["eval_precision"])
print("Recall:", test_results["eval_recall"])
print("F1:", test_results["eval_f1"])
print("Accuracy:", test_results["eval_accuracy"])

/opt/anaconda3/envs/neuralnet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.023125,0.066226,3,0.826172,0.881250,0.852823,0.983876


Precision: 0.826171875
Recall: 0.88125
F1: 0.8528225806451613
Accuracy: 0.9838755766012165


## Discussion

On the test split the baseline reached a precision of 82.62%, a recall of 88.13%, an F1 of 85.28% and a word-level accuracy of 98.39%. These are the BERT values reported in Table III of the report. Recall is clearly higher than precision, so most errors are extra or wrongly bounded predictions rather than missed mentions.

This is a single run. In Part 6 the same model was fine-tuned with three seeds and reached a test F1 of 85.43 ± 0.51%, so the run here lies within the normal seed-to-seed variation. Differences between models of less than about one F1 point should therefore be read with care. BioBERT and PubMedBERT, which are compared against this baseline in Parts 4 and 5, both score higher on the test F1.